Full precision (bf16) full finetune

What it does:
Reconstruction (optional) → CPT (full model) → SFT (full model) → Inference with compressed memory & optional expansion.


# REFRAG (Script B) — Full Precision (bf16)

Implements training + inference for REFRAG-style compressed context using LLaMA-3.1 full finetuning (bf16).

Stages: Reconstruction → CPT → SFT → Inference


In [ ]:
%pip install -q --upgrade torch transformers accelerate datasets faiss-cpu tqdm pyyaml


In [ ]:
# ======== Config (EDIT ME) ========
cfg = {
    "model_id": "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "encoder_id": "roberta-base",

    "corpus_path": "/data/cpt_corpus.jsonl",
    "sft_path": "/data/sft.jsonl",
    "passages_path": "/data/passages.jsonl",

    "out_dir_recon": "/mnt/data/ckpt_fp/recon",
    "out_dir_cpt":   "/mnt/data/ckpt_fp/cpt",
    "out_dir_sft":   "/mnt/data/ckpt_fp/sft",
    "projector_init": "/mnt/data/ckpt_fp/recon/projector.pt",
    "projector_for_infer": "/mnt/data/ckpt_fp/cpt/projector.pt",

    "prefix_len": 1,
    "chunk_size": 16,
    "max_len": 4096,
    "epochs_recon": 1,
    "epochs_cpt": 1,
    "epochs_sft": 1,
    "lr_recon": 2e-4,
    "lr_cpt": 1e-4,
    "lr_sft": 5e-5,
    "bsz_sft": 1,

    "query": "Explain root cause and remediation for this IDS alert pattern.",
    "max_new_tokens": 256,
    "expand_budget": 256,
}
print(cfg)


In [ ]:
import os, json, random
from typing import List
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoConfig, AutoTokenizer, AutoModel, AutoModelForCausalLM,
    get_cosine_schedule_with_warmup,
)
from tqdm import tqdm

def dev(): return "cuda" if torch.cuda.is_available() else "cpu"
def read_jsonl(path): return [json.loads(l) for l in open(path,"r",encoding="utf-8")]
def chunk_ids(ids, k): return [ids[i:i+k] for i in range(0,len(ids),k)]
@torch.no_grad()
""
def detok(tok, ids): return tok.decode(ids, skip_special_tokens=True)

class AttnSpec:
    def __init__(self, model_id):
        cfg=AutoConfig.from_pretrained(model_id)
        self.n_layers=int(getattr(cfg,"num_hidden_layers",0))
        self.n_q_heads=int(getattr(cfg,"num_attention_heads",0))
        self.hidden=int(getattr(cfg,"hidden_size",0))
        self.n_kv=int(getattr(cfg,"num_key_value_heads", self.n_q_heads))
        self.head_dim=int(getattr(cfg,"head_dim", self.hidden//max(1,self.n_q_heads)))

class ChunkEncoder(nn.Module):
    def __init__(self, model_id="roberta-base", device="cuda"):
        super().__init__()
        self.tok=AutoTokenizer.from_pretrained(model_id, use_fast=True)
        self.enc=AutoModel.from_pretrained(model_id)
        self.device=device; self.to(device)
    @torch.no_grad()
    def encode(self, texts: List[str], max_len=256, bs=32):
        self.eval(); outs=[]
        for i in range(0,len(texts),bs):
            b=texts[i:i+bs]
            t=self.tok(b,padding=True,truncation=True,max_length=max_len,return_tensors="pt").to(self.device)
            h=self.enc(**t).last_hidden_state
            m=t["attention_mask"].unsqueeze(-1)
            e=(h*m).sum(dim=1)/m.sum(dim=1).clamp(min=1)
            outs.append(e)
        return torch.cat(outs,dim=0)

class KVProjector(nn.Module):
    def __init__(self, enc_dim, n_layers, n_kv, head_dim, prefix_len=1, width_mult=4):
        super().__init__()
        self.nl=n_layers; self.nk=n_kv; self.hd=head_dim; self.P=prefix_len
        out_dim=n_kv*prefix_len*head_dim
        def block(): return nn.Sequential(nn.Linear(enc_dim, width_mult*enc_dim), nn.GELU(),
                                          nn.Linear(width_mult*enc_dim, out_dim))
        self.Wk=nn.ModuleList([block() for _ in range(n_layers)])
        self.Wv=nn.ModuleList([block() for _ in range(n_layers)])
    def forward(self, emb):
        B=emb.size(0)
        def shape(x): return x.view(B,self.nk,self.P,self.hd)
        K=[shape(w(emb)) for w in self.Wk]
        V=[shape(w(emb)) for w in self.Wv]
        return K,V

def broadcast_kv(kv, n_q, n_kv):
    if n_kv==n_q: return kv
    reps=n_q//n_kv
    return kv.repeat_interleave(reps,dim=1)

def build_prefix_past(K_list,V_list,spec):
    out=[]
    for l in range(spec.n_layers):
        K=broadcast_kv(K_list[l], spec.n_q_heads, spec.n_kv)
        V=broadcast_kv(V_list[l], spec.n_q_heads, spec.n_kv)
        out.append((K,V))
    return out

def concat_prefix_into_past(prefix, past):
    if past is None: return prefix
    merged=[]
    for (Kp,Vp), layer_past in zip(prefix, past):
        if layer_past is None or layer_past[0] is None: merged.append((Kp,Vp))
        else:
            K,V=layer_past
            merged.append((torch.cat([Kp,K],dim=2), torch.cat([Vp,V],dim=2)))
    return merged

class AttnThresholdPolicy:
    def __init__(self, prefix_len=1, token_budget=256, est_tokens_per_chunk=64):
        self.P=prefix_len; self.budget=token_budget; self.est=est_tokens_per_chunk
    def select(self, attn, num_chunks):
        Pn=self.P*num_chunks
        s=attn[:,:Pn].sum(dim=0)
        pairs=[]
        for i in range(num_chunks):
            pairs.append((i, s[i*self.P:(i+1)*self.P].sum().item()))
        pairs.sort(key=lambda x:x[1], reverse=True)
        sel=[]; bud=self.budget
        for i,_ in pairs:
            if bud<=0: break
            sel.append(i); bud-=self.est
        return sel


In [ ]:
# ===== Reconstruction =====
def run_reconstruction(cfg):
    spec=AttnSpec(cfg["model_id"])
    tok=AutoTokenizer.from_pretrained(cfg["model_id"], use_fast=True)
    model=AutoModelForCausalLM.from_pretrained(
        cfg["model_id"], torch_dtype=torch.bfloat16, device_map="auto"
    )
    for p in model.parameters(): p.requires_grad=False
    model.eval()

    enc=ChunkEncoder(cfg["encoder_id"], dev())
    data=read_jsonl(cfg["corpus_path"]); texts=[x["text"] for x in data]
    with torch.no_grad(): emb=enc.encode(texts)

    proj=KVProjector(emb.shape[-1],spec.n_layers,spec.n_kv,spec.head_dim, prefix_len=cfg["prefix_len"]).to(dev())
    opt=AdamW(proj.parameters(), lr=cfg["lr_recon"], weight_decay=0.01)

    bs=8
    for ep in range(cfg["epochs_recon"]):
        for i in range(0,len(texts),bs):
            btxt=texts[i:i+bs]; b_emb=emb[i:i+bs].to(dev())
            K,V=proj(b_emb); prefix=build_prefix_past(K,V,spec)
            t=tok(btxt,padding=True,truncation=True,max_length=cfg["max_len"],return_tensors="pt").to(model.device)
            lab=t["input_ids"].clone(); lab[lab==tok.pad_token_id]=-100
            out=model(**t, labels=lab, past_key_values=prefix, use_cache=True)
            loss=out.loss
            opt.zero_grad(); loss.backward(); opt.step()
            if (i//bs)%25==0: print(f"[reconstruct] ep{ep} step{i//bs} loss={loss.item():.4f}")
    os.makedirs(cfg["out_dir_recon"], exist_ok=True)
    torch.save(proj.state_dict(), os.path.join(cfg["out_dir_recon"],"projector.pt"))
    print(f"Saved projector -> {cfg['out_dir_recon']}/projector.pt")

# run_reconstruction(cfg)


In [ ]:
# ===== CPT (full finetune) =====
def run_cpt(cfg):
    spec=AttnSpec(cfg["model_id"])
    tok=AutoTokenizer.from_pretrained(cfg["model_id"], use_fast=True)
    model=AutoModelForCausalLM.from_pretrained(
        cfg["model_id"], torch_dtype=torch.bfloat16, device_map="auto"
    ).train(True)

    enc=ChunkEncoder(cfg["encoder_id"], dev())
    data=read_jsonl(cfg["corpus_path"]); texts=[x["text"] for x in data]
    with torch.no_grad(): emb=enc.encode(texts)

    proj=KVProjector(emb.shape[-1],spec.n_layers,spec.n_kv,spec.head_dim, prefix_len=cfg["prefix_len"]).to(dev())
    if os.path.exists(cfg["projector_init"]):
        proj.load_state_dict(torch.load(cfg["projector_init"], map_location=dev()))

    opt=AdamW(list(proj.parameters())+list(model.parameters()), lr=cfg["lr_cpt"], weight_decay=0.01)
    steps=cfg["epochs_cpt"]*max(1,len(texts)//8)
    sched=get_cosine_schedule_with_warmup(opt, int(0.02*steps), steps)

    bs=8
    for ep in range(cfg["epochs_cpt"]):
        for i in range(0,len(texts),bs):
            btxt=texts[i:i+bs]; b_emb=emb[i:i+bs].to(dev())
            K,V=proj(b_emb); prefix=build_prefix_past(K,V,spec)
            t=tok(btxt,padding=True,truncation=True,max_length=cfg["max_len"],return_tensors="pt").to(model.device)
            lab=t["input_ids"].clone(); lab[lab==tok.pad_token_id]=-100

            # light compression mixing
            if random.random()<0.8:
                mask=torch.rand_like(lab,dtype=torch.float)<0.5
                lab=lab.masked_fill(mask,-100)

            out=model(**t, labels=lab, past_key_values=prefix, use_cache=True)
            loss=out.loss
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            if (i//bs)%25==0: print(f"[cpt] ep{ep} step{i//bs} loss={loss.item():.4f}")

    os.makedirs(cfg["out_dir_cpt"], exist_ok=True)
    model.save_pretrained(os.path.join(cfg["out_dir_cpt"],"lm")); tok.save_pretrained(os.path.join(cfg["out_dir_cpt"],"lm"))
    torch.save(proj.state_dict(), os.path.join(cfg["out_dir_cpt"],"projector.pt"))
    print(f"Saved CPT model -> {cfg['out_dir_cpt']}/lm ; projector -> {cfg['out_dir_cpt']}/projector.pt")

# run_cpt(cfg)


In [ ]:
# ===== SFT (full finetune) =====
class SFTDS(Dataset):
    def __init__(self,path): self.items=[json.loads(l) for l in open(path,"r",encoding="utf-8")]
    def __len__(self): return len(self.items)
    def __getitem__(self,i): return self.items[i]

def run_sft(cfg):
    tok=AutoTokenizer.from_pretrained(cfg["model_id"], use_fast=True)
    model=AutoModelForCausalLM.from_pretrained(
        cfg["model_id"], torch_dtype=torch.bfloat16, device_map="auto"
    ).train(True)

    ds=SFTDS(cfg["sft_path"])
    def collate(b):
        def normalize(x):
            if "inputs" in x and "targets" in x:
                return x["inputs"], x["targets"]
            instr=(x.get("instruction","") or "").strip()
            inp=(x.get("input","") or "").strip()
            tgt=(x.get("output","") or "").strip()
            prompt = f"{instr}\n\nContext:\n{inp}" if inp else instr
            return prompt, tgt
        pairs=[normalize(x) for x in b]
        merged=[f"### Instruction\n{p}\n\n### Response\n{a}" for (p,a) in pairs]
        t=tok(merged,padding=True,truncation=True,max_length=cfg["max_len"], return_tensors="pt")
        lab=t["input_ids"].clone(); lab[lab==tok.pad_token_id]=-100
        t["labels"]=lab
        return {k:v.to(model.device) for k,v in t.items()}

    dl=DataLoader(ds,batch_size=cfg["bsz_sft"],shuffle=True,collate_fn=collate)
    opt=AdamW(model.parameters(), lr=cfg["lr_sft"], weight_decay=0.01)
    steps=cfg["epochs_sft"]*len(dl)
    sched=get_cosine_schedule_with_warmup(opt, int(0.02*steps), steps)
    for ep in range(cfg["epochs_sft"]):
        for i,b in enumerate(dl):
            out=model(**b); loss=out.loss
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            if i%50==0: print(f"[sft] ep{ep} step{i} loss={loss.item():.4f}")
    os.makedirs(cfg["out_dir_sft"], exist_ok=True)
    model.save_pretrained(cfg["out_dir_sft"]); tok.save_pretrained(cfg["out_dir_sft"])
    print(f"Saved SFT model -> {cfg['out_dir_sft']}")

# run_sft(cfg)


In [ ]:
"""
# ===== Inference =====
@torch.no_grad()
def run_infer(cfg):
    spec=AttnSpec(cfg["model_id"])
    tok=AutoTokenizer.from_pretrained(cfg["model_id"], use_fast=True)
    model=AutoModelForCausalLM.from_pretrained(
        cfg["model_id"], torch_dtype=torch.bfloat16, device_map="auto", output_attentions=True
    ).eval()

    enc=ChunkEncoder(cfg["encoder_id"], dev())
    passages=read_jsonl(cfg["passages_path"])
    chunk_texts=[]
    for p in passages:
        ids=tok(p["text"], add_special_tokens=False, return_tensors="pt")["input_ids"][0].tolist()
        for c in chunk_ids(ids, cfg["chunk_size"]):
            s=detok(tok,c).strip()
            if s: chunk_texts.append(s)

    emb=enc.encode(chunk_texts)
    proj=KVProjector(emb.shape[-1], spec.n_layers, spec.n_kv, spec.head_dim, prefix_len=cfg["prefix_len"]).to(dev())
    if os.path.exists(cfg["projector_for_infer"]):
        proj.load_state_dict(torch.load(cfg["projector_for_infer"], map_location=dev()))
    K,V=proj(emb.to(dev()))
    prefix=build_prefix_past(K,V,spec)

    prompt=( "You are a networking & security assistant. Use compressed memory to answer precisely.\n\n"
             f"Question: {cfg['query']}\nAnswer:" )
    inp=tok(prompt, return_tensors="pt").to(model.device)
    past=concat_prefix_into_past(prefix,None)
    policy=AttnThresholdPolicy(prefix_len=cfg["prefix_len"], token_budget=cfg["expand_budget"], est_tokens_per_chunk=cfg["chunk_size"])

    out_ids=[]; cur=inp["input_ids"]
    for step in range(cfg["max_new_tokens"]):
        out=model(input_ids=cur, use_cache=True, past_key_values=past, output_attentions=True)
        nxt=torch.argmax(out.logits[:,-1,:],dim=-1,keepdim=True)
        out_ids.append(nxt)
        attn=None
        if out.attentions:
            for la in out.attentions:
                B,H,Tq,Tk=la.shape
                Pn=prefix[0][0].shape[2]
                if Tk>=Pn:
                    attn=la[0,:, -1, :Pn]
                    break
        if attn is not None and cfg["expand_budget"]>0 and step in (32,96):
            sel=policy.select(attn, num_chunks=len(chunk_texts))
            if sel:
                exp="\n\n".join(chunk_texts[i] for i in sel)
                e=tok(exp, return_tensors="pt", add_special_tokens=False).to(model.device)
                cur=e["input_ids"]; cfg["expand_budget"]-=int(cur.numel()); continue
        cur=nxt
        if tok.eos_token_id is not None and int(nxt.item())==tok.eos_token_id: break

    text=tok.decode(torch.cat(out_ids,dim=-1)[0], skip_special_tokens=True)
    print(text)

# run_infer(cfg)


"""

"""
inference runs in a token-by-token loop.

Quick breakdown of what that loop does in the notebooks:

Generates one token at a time until either:

it hits cfg["max_new_tokens"], or

the model emits the EOS token.

Uses your compressed memory prefix (past_key_values built from the projector) on every step.

Optional selective expansion: at specific steps (in the code: step in (32, 96)), it peeks at attention over the prefix, picks the top chunks, and injects their raw text (spending from cfg["expand_budget"]). After injecting, it continues token-by-token generation.

Stops when EOS is generated or the token limit is reached.

Tweak points (if you want different behavior):

Total response length: change cfg["max_new_tokens"].

Whether/when to expand: edit the tuple (32, 96) or remove that block.

How much expansion is allowed: change cfg["expand_budget"].

Granularity of chunking: change cfg["chunk_size"].

Disable expansion entirely: set cfg["expand_budget"]=0 or comment out the expansion block.

So yes, it’s a standard autoregressive decode loop with optional attention-guided expansion hooks.

"""